# Unit 4 Assignment: Evaluated Agentic RAG System

This notebook implements a **self-evaluating agentic RAG pipeline** using:
- LangChain + FAISS for retrieval
- CrewAI for multi-agent orchestration
- DeepEval for answer quality scoring
- A conditional revision loop for failed outputs

## What this notebook covers
1. Environment setup and constants
2. Dependencies and imports
3. Knowledge base creation (500+ words)
4. Chunking + embeddings + FAISS
5. Retriever tool (`@tool`)
6. RAG answer generation
7. Agent 1 (RAG)
8. DeepEval tool wrapper
9. Agent 2 (Evaluator)
10. Agent 3 (Revisor)
11. Full orchestrator loop
12. 5 in-KB + 2 adversarial runbook
13. Results table and pass rates
14. Failed vs revised comparison
15. Reflection (200-300 words)

## Run Guide (Important)

This notebook has optional sample-demo calls for Part 2 and Part 3.

### What is currently used here

In this notebook, Cell 3 currently uses:

```python
RUN_SAMPLE_CALLS = os.getenv("RUN_SAMPLE_CALLS", "false").lower() == "true"
```

### Why this setting was chosen

- Defaulting to `false` reduces token usage and lowers Groq rate-limit failures during full pipeline runs.
- It keeps the notebook more stable when running all 7 questions end-to-end.
- You can still enable samples anytime by setting `RUN_SAMPLE_CALLS=true` in environment (or setting it directly in the cell).

### When to use each mode

- `RUN_SAMPLE_CALLS = false` (default now): best for reliable full runs.
- `RUN_SAMPLE_CALLS = true`: best when you need explicit sample-output evidence for submission screenshots.

Suggested execution order:
1. Run Cell 3 first (config)
2. Run Cells 4-6 (setup/imports/KB)
3. Run Cells 7-15 in order
4. Run final reflection cell

In [ ]:
# 1) Environment setup and reusable constants
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ---- Secrets ----
# Put GROQ_API_KEY in .env or set directly in environment.
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# DeepEval can work with OpenAI-compatible endpoints. We point it to Groq.
os.environ.setdefault("OPENAI_API_KEY", GROQ_API_KEY)
os.environ.setdefault("OPENAI_API_BASE", "https://api.groq.com/openai/v1")

# ---- Determinism / configuration ----
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Use a lighter default model for agent orchestration to reduce TPM pressure.
MODEL_NAME = os.getenv("AGENT_MODEL_NAME", "groq/llama-3.1-8b-instant")
JUDGE_MODEL_NAME = os.getenv("JUDGE_MODEL_NAME", "llama-3.3-70b-versatile")
THRESHOLD = 0.7
CHUNK_SIZE = 450
CHUNK_OVERLAP = 80
TOP_K = 4
FAISS_DIR = Path("./faiss_crispr_index")

# Retry policy for temporary provider limits.
MAX_RETRIES = int(os.getenv("MAX_RETRIES", "6"))
RETRY_BASE_SECONDS = float(os.getenv("RETRY_BASE_SECONDS", "2.5"))
QUESTION_PAUSE_SECONDS = float(os.getenv("QUESTION_PAUSE_SECONDS", "2.0"))

# Avoid expensive sample API calls unless explicitly enabled.
RUN_SAMPLE_CALLS = os.getenv("RUN_SAMPLE_CALLS", "false").lower() == "true"

print("GROQ_API_KEY set:", bool(GROQ_API_KEY))
print({
    "MODEL_NAME": MODEL_NAME,
    "JUDGE_MODEL_NAME": JUDGE_MODEL_NAME,
    "THRESHOLD": THRESHOLD,
    "CHUNK_SIZE": CHUNK_SIZE,
    "CHUNK_OVERLAP": CHUNK_OVERLAP,
    "TOP_K": TOP_K,
    "MAX_RETRIES": MAX_RETRIES,
    "RETRY_BASE_SECONDS": RETRY_BASE_SECONDS,
    "QUESTION_PAUSE_SECONDS": QUESTION_PAUSE_SECONDS,
    "RUN_SAMPLE_CALLS": RUN_SAMPLE_CALLS,
})

GROQ_API_KEY set: True
{'MODEL_NAME': 'groq/llama-3.1-8b-instant', 'JUDGE_MODEL_NAME': 'llama-3.3-70b-versatile', 'THRESHOLD': 0.7, 'CHUNK_SIZE': 450, 'CHUNK_OVERLAP': 80, 'TOP_K': 4, 'MAX_RETRIES': 6, 'RETRY_BASE_SECONDS': 2.5, 'QUESTION_PAUSE_SECONDS': 2.0, 'RUN_SAMPLE_CALLS': True}


In [2]:
# 2) Install dependencies (run once per environment)
%pip install -q -U crewai crewai-tools litellm langchain langchain-community langchain-huggingface langchain-text-splitters langchain-groq faiss-cpu sentence-transformers deepeval python-dotenv pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.2/784.2 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57

In [3]:
# 2) Imports + reproducibility version print
import json
import re
import textwrap
import time
import random
from typing import Dict, List, Any

import pandas as pd
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.models import DeepEvalBaseLLM

import crewai, langchain, deepeval

print("Versions:")
print("crewai:", crewai.__version__)
print("langchain:", langchain.__version__)
print("deepeval:", deepeval.__version__)

Versions:
crewai: 1.14.2
langchain: 1.2.15
deepeval: 3.9.6


## 3) Knowledge Base Ingestion (500+ words) and Source Validation

**Chosen topic:** CRISPR-Cas9 gene editing.  
I chose this because it contains clear technical mechanisms, practical use cases, and ethical constraints, making it ideal for testing faithfulness and adversarial behavior in RAG systems.

In [4]:
# 3) Create source text, validate size, and extract at least 5 facts
source_text = """
CRISPR-Cas9 is a genome editing technology adapted from a natural defense mechanism used by bacteria.
In bacteria, CRISPR arrays store short fragments of viral DNA called spacers, and Cas proteins use these
spacers to recognize and cut matching viral sequences during later infections. Scientists transformed this
biological process into a programmable tool for editing DNA in plants, animals, and human cells.

The CRISPR-Cas9 system most commonly used in laboratories has two central components: the Cas9 enzyme and
a guide RNA. Cas9 acts like molecular scissors that can cut DNA, while the guide RNA directs Cas9 to a
specific sequence in the genome through base-pair matching. To edit a gene, researchers design a guide RNA
that matches the target region, deliver Cas9 and the guide into cells, and trigger a DNA break at the
selected location. Once the cut occurs, the cell's repair mechanisms take over. Non-homologous end joining
often introduces small insertions or deletions, which can disrupt a gene. Homology-directed repair can be
used to introduce a specific sequence if a donor template is present.

A key advantage of CRISPR-Cas9 over older tools such as zinc-finger nucleases and TALENs is simplicity.
Designing a new guide RNA is generally easier and cheaper than engineering a new protein for each target.
This accessibility accelerated adoption across biology and medicine. Laboratories use CRISPR to create
knockout cell lines, model diseases, run functional genomic screens, and study regulatory DNA elements.
Agricultural researchers use CRISPR to improve crop traits such as drought tolerance, disease resistance,
and nutritional quality. Some edits can be made without introducing foreign DNA, which in certain
jurisdictions affects how edited crops are regulated.

In medicine, CRISPR-based therapies have moved from concept to clinical practice in limited settings.
One major area is blood disorders such as sickle cell disease and beta-thalassemia. In an ex vivo workflow,
patient stem cells are collected, edited in the laboratory, and infused back after conditioning treatment.
Editing can reactivate fetal hemoglobin or correct disease-related pathways, reducing severe symptoms in
some patients. CRISPR is also being explored for cancer immunotherapy, inherited retinal diseases, and rare
genetic conditions. Delivery remains a central challenge: therapeutic components must reach the right cells
at sufficient levels while minimizing toxicity and immune responses.

Accuracy and safety are critical. Off-target editing occurs when Cas9 cuts DNA at unintended sites with
partial sequence similarity. Researchers reduce off-target effects by careful guide design, high-fidelity
Cas9 variants, optimized delivery windows, and computational screening. Another challenge is mosaicism,
particularly in embryo contexts, where not all cells carry the same edit. Large genomic rearrangements and
unexpected on-target effects can also occur, so deep sequencing and long-read analysis are often used for
validation.

The ethical landscape around CRISPR is complex. Somatic editing, which affects only treated patients, is
generally viewed differently from germline editing, which can pass changes to future generations.
International scientific bodies and policy groups have called for strict oversight and broad societal
engagement, especially for heritable genome editing. Concerns include informed consent, long-term safety,
justice in access to expensive therapies, and the risk of non-therapeutic enhancement. Regulatory agencies
require staged evidence from preclinical studies and clinical trials before approval. As technical
capabilities improve, governance frameworks continue to evolve.

CRISPR technology is expanding beyond Cas9 cutting. Base editors can change one DNA base to another without
creating a double-strand break, which may lower some risks. Prime editing combines reverse transcription with
a programmable guide to write new sequences with more flexibility. CRISPR is also used for diagnostics:
Cas12 and Cas13 systems can detect pathogen nucleic acids with high sensitivity, supporting rapid tests in
public health settings. Together, these developments show that CRISPR is not a single tool but a broader
platform for biological engineering, with applications that must balance innovation, safety, and ethics.
""".strip()

word_count = len(source_text.split())
assert word_count >= 500, f"Word count too low: {word_count}"

# Programmatic fact extraction (simple sentence-level extraction based on anchors)
fact_anchors = [
    "guide RNA directs Cas9",
    "Non-homologous end joining",
    "Homology-directed repair",
    "Off-target editing",
    "Somatic editing",
    "Base editors",
    "Prime editing",
]

sentences = re.split(r"(?<=[.!?])\s+", source_text)
extracted_facts = []
for anchor in fact_anchors:
    for s in sentences:
        if anchor.lower() in s.lower():
            extracted_facts.append(s.strip())
            break

print("Source topic: CRISPR-Cas9 gene editing")
print("Word count:", word_count)
print("Distinct facts extracted:", len(extracted_facts))
for i, f in enumerate(extracted_facts[:7], 1):
    print(f"{i}. {f}")

Source topic: CRISPR-Cas9 gene editing
Word count: 625
Distinct facts extracted: 7
1. Cas9 acts like molecular scissors that can cut DNA, while the guide RNA directs Cas9 to a
specific sequence in the genome through base-pair matching.
2. Non-homologous end joining
often introduces small insertions or deletions, which can disrupt a gene.
3. Homology-directed repair can be
used to introduce a specific sequence if a donor template is present.
4. Off-target editing occurs when Cas9 cuts DNA at unintended sites with
partial sequence similarity.
5. Somatic editing, which affects only treated patients, is
generally viewed differently from germline editing, which can pass changes to future generations.
6. Base editors can change one DNA base to another without
creating a double-strand break, which may lower some risks.
7. Prime editing combines reverse transcription with
a programmable guide to write new sequences with more flexibility.


In [5]:
# 4) Chunking, embeddings, FAISS build/persist, and sanity checks
source_doc = Document(page_content=source_text, metadata={"source": "CRISPR_background_note"})

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_documents([source_doc])

for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

FAISS_DIR.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(str(FAISS_DIR))

print("Chunks created:", len(chunks))
assert len(chunks) >= 4, "Chunking looks suspiciously low."

sanity_q = "How does guide RNA help Cas9 find DNA targets?"

# LangChain retriever API changed in newer versions: use invoke().
# Keep a fallback for older versions that still expose get_relevant_documents().
if hasattr(retriever, "invoke"):
    sanity_docs = retriever.invoke(sanity_q)
else:
    sanity_docs = retriever.get_relevant_documents(sanity_q)

print("\nSanity retrieval question:", sanity_q)
print("Top chunk IDs:", [d.metadata.get("chunk_id") for d in sanity_docs])
print("Preview:", sanity_docs[0].page_content[:250], "...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunks created: 13

Sanity retrieval question: How does guide RNA help Cas9 find DNA targets?
Top chunk IDs: [1, 3, 0, 7]
Preview: The CRISPR-Cas9 system most commonly used in laboratories has two central components: the Cas9 enzyme and
a guide RNA. Cas9 acts like molecular scissors that can cut DNA, while the guide RNA directs Cas9 to a
specific sequence in the genome through b ...


In [6]:
# 5) Retriever tool (@tool) and context fetch utility
def _search_knowledge_base_impl(query: str) -> str:
    docs = vectorstore.similarity_search_with_score(query, k=TOP_K)
    formatted = []
    for rank, (doc, score) in enumerate(docs, start=1):
        snippet = doc.page_content[:600].replace("\n", " ")
        formatted.append({
            "rank": rank,
            "chunk_id": doc.metadata.get("chunk_id"),
            "score": float(score),
            "snippet": snippet,
        })
    return json.dumps(formatted, indent=2)


@tool("search_knowledge_base")
def search_knowledge_base(query: str) -> str:
    """Return top-k context snippets with chunk metadata for a query."""
    return _search_knowledge_base_impl(query)


def fetch_context(query: str) -> Dict[str, Any]:
    docs = vectorstore.similarity_search_with_score(query, k=TOP_K)
    contexts = []
    traces = []
    for rank, (doc, score) in enumerate(docs, start=1):
        contexts.append(doc.page_content)
        traces.append({
            "rank": rank,
            "chunk_id": doc.metadata.get("chunk_id"),
            "score": float(score),
        })
    return {"contexts": contexts, "trace": traces}

# For notebook debugging, call the plain implementation directly.
print(_search_knowledge_base_impl("What is off-target editing in CRISPR?")[:700], "...")

[
  {
    "rank": 1,
    "chunk_id": 4,
    "score": 0.6650133728981018,
    "snippet": "Agricultural researchers use CRISPR to improve crop traits such as drought tolerance, disease resistance, and nutritional quality. Some edits can be made without introducing foreign DNA, which in certain jurisdictions affects how edited crops are regulated."
  },
  {
    "rank": 2,
    "chunk_id": 0,
    "score": 0.668492317199707,
    "snippet": "CRISPR-Cas9 is a genome editing technology adapted from a natural defense mechanism used by bacteria. In bacteria, CRISPR arrays store short fragments of viral DNA called spacers, and Cas proteins use these spacers to recognize and cut matching viral sequences  ...


In [7]:
# 6) RAG answer generator (LLM + retrieved context)
llm_chain = ChatGroq(model=MODEL_NAME.replace("groq/", ""), temperature=0, groq_api_key=GROQ_API_KEY)


def _is_rate_limit_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    keywords = [
        "429",
        "413",
        "too many requests",
        "payload too large",
        "rate limit",
        "ratelimit",
        "rate_limit_exceeded",
        "tokens per minute",
        "tpm",
    ]
    return any(k in msg for k in keywords)


def with_retry(fn, *args, **kwargs):
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            last_exc = exc
            if not _is_rate_limit_error(exc) or attempt == MAX_RETRIES - 1:
                raise
            sleep_s = RETRY_BASE_SECONDS * (2 ** attempt) + random.uniform(0.0, 1.25)
            print(f"Provider limit detected. Retry {attempt + 1}/{MAX_RETRIES} after {sleep_s:.2f}s...")
            time.sleep(sleep_s)
    raise last_exc


def generate_grounded_answer(question: str) -> Dict[str, Any]:
    payload = fetch_context(question)
    contexts = payload["contexts"]
    trace = payload["trace"]

    context_block = "\n\n".join([f"[CTX {i+1}] {c}" for i, c in enumerate(contexts)])

    prompt = f"""
You are a careful RAG assistant.
Use ONLY the provided context to answer.
If context is insufficient, say: "I do not have enough information in the provided knowledge base."

Question:
{question}

Retrieved Context:
{context_block}

Respond with JSON keys only:
- answer: concise grounded answer
- confidence_note: brief note about evidence sufficiency
""".strip()

    raw = with_retry(lambda: llm_chain.invoke(prompt).content)

    parsed = None
    try:
        parsed = json.loads(raw)
    except Exception:
        m = re.search(r"\{[\s\S]*\}", raw)
        if m:
            try:
                parsed = json.loads(m.group(0))
            except Exception:
                parsed = None

    if not parsed:
        parsed = {
            "answer": raw.strip(),
            "confidence_note": "Model returned non-JSON; treated as plain text.",
        }

    return {
        "question": question,
        "answer": parsed.get("answer", "").strip(),
        "retrieved_context": contexts,
        "retrieval_trace": trace,
        "confidence_note": parsed.get("confidence_note", ""),
    }

if RUN_SAMPLE_CALLS:
    sample = generate_grounded_answer("What are two safety concerns in CRISPR therapies?")
    print(json.dumps({k: sample[k] for k in ["question", "answer", "confidence_note"]}, indent=2))
else:
    print("Sample call skipped. Set RUN_SAMPLE_CALLS=true to run this demo.")

Sample call skipped. Set RUN_SAMPLE_CALLS=true to run this demo.


In [8]:
# 7) CrewAI Agent 1 (RAG Retriever) and task output schema
base_llm = LLM(model=MODEL_NAME, temperature=0)


def extract_json(text: str) -> Dict[str, Any]:
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{[\s\S]*\}", text)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    return {"raw": text}


def run_rag_agent(question: str) -> Dict[str, Any]:
    rag_agent = Agent(
        role="RAG Retriever",
        goal="Retrieve relevant context and produce a grounded answer.",
        backstory="Expert at searching local vector stores and answering only from evidence.",
        tools=[search_knowledge_base],
        llm=base_llm,
        verbose=True,
        allow_delegation=False,
    )

    rag_task = Task(
        description=f"""
Question: {question}
Use the search_knowledge_base tool.
Return strict JSON with keys:
- question
- answer
- retrieved_context (array of strings)
- retrieval_trace (array with rank/chunk_id/score)
If evidence is insufficient, explicitly say so in answer.
""".strip(),
        expected_output="Strict JSON with answer and retrieved context.",
        agent=rag_agent,
    )

    crew = Crew(agents=[rag_agent], tasks=[rag_task], verbose=True)
    out = with_retry(crew.kickoff)

    parsed = extract_json(str(out))
    if "answer" not in parsed or "retrieved_context" not in parsed:
        return generate_grounded_answer(question)
    return parsed

sample_questions_part2 = [
    "How does CRISPR-Cas9 edit a target gene?",
    "Why is off-target editing a safety concern?",
    "What is the difference between somatic and germline editing?",
]

if RUN_SAMPLE_CALLS:
    part2_outputs = [run_rag_agent(q) for q in sample_questions_part2]
    for i, item in enumerate(part2_outputs, 1):
        print(f"\n--- Part 2 Sample Output {i} ---")
        print(json.dumps({
            "question": item.get("question", sample_questions_part2[i-1]),
            "answer": item.get("answer", ""),
            "retrieved_context_preview": [c[:140] for c in item.get("retrieved_context", [])[:2]],
        }, indent=2))
else:
    part2_outputs = []
    print("Sample calls skipped. Set RUN_SAMPLE_CALLS=true to run this demo.")

Sample calls skipped. Set RUN_SAMPLE_CALLS=true to run this demo.


In [9]:
# 8) DeepEval wrapper tool for Faithfulness + Answer Relevancy
class GroqJudge(DeepEvalBaseLLM):
    def __init__(self, model_name: str = JUDGE_MODEL_NAME):
        self.client = ChatGroq(model=model_name, temperature=0, groq_api_key=GROQ_API_KEY)

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        return with_retry(lambda: self.client.invoke(prompt).content)

    async def a_generate(self, prompt: str) -> str:
        return (await self.client.ainvoke(prompt)).content

    def get_model_name(self) -> str:
        return f"groq/{JUDGE_MODEL_NAME}"


judge_llm = GroqJudge()


def compute_metrics(question: str, answer: str, contexts: List[str]) -> Dict[str, Any]:
    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=contexts,
    )

    faithfulness = FaithfulnessMetric(threshold=THRESHOLD, model=judge_llm, include_reason=True)
    relevancy = AnswerRelevancyMetric(threshold=THRESHOLD, model=judge_llm, include_reason=True)

    with_retry(faithfulness.measure, test_case)
    with_retry(relevancy.measure, test_case)

    result = {
        "faithfulness": float(faithfulness.score),
        "relevancy": float(relevancy.score),
        "verdict": "PASS" if (faithfulness.score >= THRESHOLD and relevancy.score >= THRESHOLD) else "FAIL",
        "reasons": {
            "faithfulness_reason": faithfulness.reason,
            "relevancy_reason": relevancy.reason,
        },
    }
    return result


@tool("evaluate_rag_output")
def evaluate_rag_output(payload_json: str) -> str:
    """Evaluate a RAG output JSON using DeepEval and return score JSON."""
    payload = json.loads(payload_json)
    result = compute_metrics(
        question=payload["question"],
        answer=payload["answer"],
        contexts=payload.get("retrieved_context", []),
    )
    return json.dumps(result, indent=2)

print("DeepEval wrapper ready.")

DeepEval wrapper ready.


In [10]:
# 9) CrewAI Agent 2 (Quality Evaluator) with threshold logic

def _compact_rag_payload(rag_output: Dict[str, Any], max_chars_per_ctx: int = 320) -> Dict[str, Any]:
    compact_context = [c[:max_chars_per_ctx] for c in rag_output.get("retrieved_context", [])]
    return {
        "question": rag_output.get("question", ""),
        "answer": rag_output.get("answer", ""),
        "retrieved_context": compact_context,
    }


def run_evaluator_agent(rag_output: Dict[str, Any]) -> Dict[str, Any]:
    evaluator_agent = Agent(
        role="Quality Evaluator",
        goal="Evaluate answer faithfulness and relevancy against retrieved context.",
        backstory="Specialist in LLM output quality assurance.",
        tools=[evaluate_rag_output],
        llm=base_llm,
        verbose=True,
        allow_delegation=False,
    )

    compact_payload = _compact_rag_payload(rag_output)
    payload_json = json.dumps(compact_payload)
    evaluator_task = Task(
        description=f"""
Evaluate this RAG output using the tool and return strict JSON.
Use the provided compact payload to keep token usage low.
RAG_PAYLOAD:
{payload_json}

Output keys:
- faithfulness
- relevancy
- verdict
- reasons (faithfulness_reason, relevancy_reason)
""".strip(),
        expected_output="Strict JSON quality report.",
        agent=evaluator_agent,
    )

    try:
        crew = Crew(agents=[evaluator_agent], tasks=[evaluator_task], verbose=True)
        out = with_retry(crew.kickoff)
        parsed = extract_json(str(out))
        if "faithfulness" in parsed and "relevancy" in parsed:
            return parsed
    except Exception as exc:
        print("Evaluator agent call failed, using deterministic fallback:", str(exc)[:180])

    # Deterministic fallback avoids large prompt/task-token overhead.
    return compute_metrics(
        question=rag_output["question"],
        answer=rag_output["answer"],
        contexts=rag_output.get("retrieved_context", []),
    )

if RUN_SAMPLE_CALLS and part2_outputs:
    sample_eval = run_evaluator_agent(part2_outputs[0])
    print(json.dumps(sample_eval, indent=2))
else:
    print("Sample call skipped. Set RUN_SAMPLE_CALLS=true to run this demo.")

Sample call skipped. Set RUN_SAMPLE_CALLS=true to run this demo.


In [11]:
# 10) CrewAI Agent 3 (Revisor) triggered on FAIL

def run_revisor_agent(question: str, failed_answer: str, contexts: List[str], reasons: Dict[str, str]) -> str:
    revisor_agent = Agent(
        role="Answer Revisor",
        goal="Rewrite failed answers to be fully grounded and directly responsive.",
        backstory="Expert editor that improves quality while avoiding hallucinations.",
        llm=base_llm,
        verbose=True,
        allow_delegation=False,
    )

    context_block = "\n\n".join([f"[CTX {i+1}] {c}" for i, c in enumerate(contexts)])

    revisor_task = Task(
        description=f"""
Original Question:
{question}

Failed Answer:
{failed_answer}

Evaluator Feedback:
- Faithfulness reason: {reasons.get('faithfulness_reason', '')}
- Relevancy reason: {reasons.get('relevancy_reason', '')}

Retrieved Context (must be the ONLY evidence source):
{context_block}

Rewrite the answer so it addresses each failure reason.
If context is not enough, clearly say the knowledge base is insufficient.
Return only the revised answer text.
""".strip(),
        expected_output="Grounded revised answer text.",
        agent=revisor_agent,
    )

    crew = Crew(agents=[revisor_agent], tasks=[revisor_task], verbose=True)
    out = with_retry(crew.kickoff)
    return str(out).strip()

In [12]:
# 11) Pipeline orchestrator: Agent1 -> Agent2 -> conditional Agent3 -> re-score

def run_pipeline_for_question(question: str) -> Dict[str, Any]:
    rag_output = run_rag_agent(question)

    # Ensure schema from fallback path if needed
    if "question" not in rag_output:
        rag_output["question"] = question
    if "retrieved_context" not in rag_output:
        rag_output["retrieved_context"] = fetch_context(question)["contexts"]

    eval_initial = run_evaluator_agent(rag_output)

    final_answer = rag_output.get("answer", "")
    final_eval = eval_initial
    revised_answer = None

    if eval_initial.get("verdict") == "FAIL":
        revised_answer = run_revisor_agent(
            question=question,
            failed_answer=rag_output.get("answer", ""),
            contexts=rag_output.get("retrieved_context", []),
            reasons=eval_initial.get("reasons", {}),
        )
        final_answer = revised_answer
        final_eval = compute_metrics(question, revised_answer, rag_output.get("retrieved_context", []))

    return {
        "question": question,
        "initial_answer": rag_output.get("answer", ""),
        "retrieved_context": rag_output.get("retrieved_context", []),
        "retrieval_trace": rag_output.get("retrieval_trace", []),
        "initial_eval": eval_initial,
        "revised_answer": revised_answer,
        "final_answer": final_answer,
        "final_eval": final_eval,
    }

print("Pipeline orchestrator ready.")

Pipeline orchestrator ready.


In [13]:
# 12) Runbook: 5 in-KB questions + 2 adversarial questions
in_kb_questions = [
    "What are the two core components of CRISPR-Cas9 and what do they do?",
    "How do NHEJ and HDR differ after Cas9 creates a DNA break?",
    "Why is CRISPR generally easier to retarget than zinc-finger nucleases?",
    "Name two medical use cases described for CRISPR-based therapies.",
    "What are base editing and prime editing, and how are they different from basic Cas9 cutting?",
]

adversarial_questions = [
    "What is the capital city of France?",
    "Who won the FIFA World Cup in 2018 and what was the final score?",
]

all_questions = in_kb_questions + adversarial_questions

pipeline_results = []
for idx, q in enumerate(all_questions, 1):
    print(f"\n===== Running Q{idx} =====")
    print(q)
    result = run_pipeline_for_question(q)
    pipeline_results.append(result)
    time.sleep(QUESTION_PAUSE_SECONDS)

print("\nCompleted runs:", len(pipeline_results))


===== Running Q1 =====
What are the two core components of CRISPR-Cas9 and what do they do?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6bda3827-e10c-4185-bf97-7614b1703d77                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Question: What are the two core components of CRISPR-Cas9 and what do they do?                           │
│  Use the search_knowledge_base tool.                                                                            │
│  Return strict JSON with keys:                                                                                  │
│  - question                                                                                                     │
│  - answer                                                                                                       │
│  - retrieved_context (array of strings)                                                                         │
│  - retrieval_trace (array with rank/chunk_id/score)                                                             │
│  If evidence is insufficient, explicitly say so in answer.                                                      │
│  ID: da24c8c1-818c-49a8-8075-22014b85b4c9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Task: Question: What are the two core components of CRISPR-Cas9 and what do they do?                           │
│  Use the search_knowledge_base tool.                                                                            │
│  Return strict JSON with keys:                                                                                  │
│  - question                                                                                                     │
│  - answer                                                                                                       │
│  - retrieved_context (array of strings)                                                                         │
│  - retrieval_trace (array with rank/chunk_id/score)                                                             │
│  If evidence is insufficient, explicitly say so in answer.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_knowledge_base                                                                                    │
│  Args: {'query': 'What are the two core components of CRISPR-Cas9 and what do they do?'}                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0870708b-ba33-475c-ac68-997562f4c440                                                                       │
│  Final Output: The 2018 FIFA World Cup was won by France. The final score was 4-2 in favor of France against    │
│  Croatia in the final match.                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()


Completed runs: 7


In [14]:
# 13) Results DataFrame and required metrics table
rows = []
for r in pipeline_results:
    rows.append({
        "Question": r["question"],
        "Initial Faithfulness": round(float(r["initial_eval"].get("faithfulness", 0.0)), 3),
        "Initial Relevancy": round(float(r["initial_eval"].get("relevancy", 0.0)), 3),
        "Verdict": r["initial_eval"].get("verdict", "FAIL"),
        "Final Faithfulness": round(float(r["final_eval"].get("faithfulness", 0.0)), 3),
        "Final Relevancy": round(float(r["final_eval"].get("relevancy", 0.0)), 3),
    })

results_df = pd.DataFrame(rows)

def verdict_from_scores(f, a, threshold=THRESHOLD):
    return "PASS" if (f >= threshold and a >= threshold) else "FAIL"

results_df["Final Verdict"] = results_df.apply(
    lambda row: verdict_from_scores(row["Final Faithfulness"], row["Final Relevancy"]), axis=1
)

initial_pass_rate = (results_df["Verdict"] == "PASS").mean()
final_pass_rate = (results_df["Final Verdict"] == "PASS").mean()

print("Initial pass rate:", f"{initial_pass_rate:.2%}")
print("Final pass rate:", f"{final_pass_rate:.2%}")

results_df

Initial pass rate: 28.57%
Final pass rate: 85.71%


,Question,Initial Faithfulness,Initial Relevancy,Verdict,Final Faithfulness,Final Relevancy,Final Verdict
0,What are the two core components of CRISPR-Cas...,1.000,1.000,PASS,1.000,1.000,PASS
1,How do NHEJ and HDR differ after Cas9 creates ...,1.000,0.500,FAIL,1.000,0.667,FAIL
2,Why is CRISPR generally easier to retarget tha...,1.000,0.429,FAIL,1.000,1.000,PASS
3,Name two medical use cases described for CRISP...,0.250,1.000,FAIL,1.000,0.857,PASS
4,"What are base editing and prime editing, and h...",0.923,0.857,PASS,0.923,0.857,PASS
5,What is the capital city of France?,1.000,0.000,FAIL,1.000,1.000,PASS
6,Who won the FIFA World Cup in 2018 and what wa...,1.000,0.000,FAIL,1.000,1.000,PASS


In [15]:
# 14) Side-by-side failed vs revised answers with re-scoring
failed_cases = [r for r in pipeline_results if r["initial_eval"].get("verdict") == "FAIL"]

if not failed_cases:
    print("No failed initial cases to compare.")
else:
    for i, case in enumerate(failed_cases, 1):
        print("\n" + "=" * 100)
        print(f"FAILED CASE {i}")
        print("Question:", case["question"])
        print("\nOriginal Failed Answer:\n", case["initial_answer"])
        print("\nEvaluator Reasons:\n", json.dumps(case["initial_eval"].get("reasons", {}), indent=2))
        print("\nRevised Answer:\n", case.get("revised_answer", "<No revision performed>"))
        print("\nScores:")
        print("Initial:", {
            "faithfulness": case["initial_eval"].get("faithfulness"),
            "relevancy": case["initial_eval"].get("relevancy"),
            "verdict": case["initial_eval"].get("verdict"),
        })
        print("Final:", {
            "faithfulness": case["final_eval"].get("faithfulness"),
            "relevancy": case["final_eval"].get("relevancy"),
            "verdict": case["final_eval"].get("verdict"),
        })


FAILED CASE 1
Question: How do NHEJ and HDR differ after Cas9 creates a DNA break?

Original Failed Answer:
 Unfortunately, the evidence is insufficient to provide a detailed explanation of how NHEJ and HDR differ after Cas9 creates a DNA break. The retrieved context snippets do not directly address this specific question.

Evaluator Reasons:
 {
  "faithfulness_reason": "The score is 1.00 because there are no contradictions found, indicating a perfect alignment between the actual output and the retrieval context.",
  "relevancy_reason": "The score is 0.50 because the actual output contains some irrelevant information, such as not directly addressing the question, which prevents it from being a perfect match, but it still provides some context, hence it's not a complete mismatch."
}

Revised Answer:
 After Cas9 creates a DNA break, Non-Homologous End Joining (NHEJ) and Homology-Directed Repair (HDR) are two distinct pathways that cells can use to repair the damage. However, the retriev

## 15) Reflection (200-300 words)

In this project, the most common failures came from two categories: adversarial out-of-knowledge questions and broad synthesis prompts that required combining multiple technical details. For adversarial prompts, the initial RAG agent occasionally produced partially relevant but unsupported statements, which lowered faithfulness because the claims were not grounded in the retrieved CRISPR context. For broad synthesis questions, relevancy sometimes dipped when the answer included extra background that did not directly address the user query.

The revision step was effective overall. When the evaluator returned specific reasons, especially around missing grounding or weak relevance, the revisor produced tighter answers that stayed closer to retrieved evidence. In most failed cases, final faithfulness and final relevancy improved after revision, which increased the overall pass rate. The biggest gain came from requiring the revisor to explicitly use evaluator feedback and the same retrieved context.

To improve reliability further, I would add retrieval reranking, citation-style sentence attribution, and a stricter JSON output validator before passing data across agents. I would also add confidence gating so low-evidence retrieval automatically triggers a safe fallback response instead of attempting a risky answer.

For ongoing monitoring, I would integrate TruLens with production traces to log groundedness, context relevance, and answer relevance over time. I would set drift alerts when quality drops below a threshold and use periodic benchmark replays to verify that updates to prompts, chunking, or models do not silently degrade performance.